In [26]:
from svector import svector
a = svector()
a


svector(float, {})

In [27]:
a['the'] = 1
a

svector(float, {'the': 1})

# Given Code

In [28]:
from __future__ import division
import time
import pandas as pd
from svector import svector

def read_from(textfile):
    data = pd.read_csv(textfile)
    for i in range(len(data)):
        id, words, label = data.iloc[i]
        yield (1 if label=="+" else -1, words.split())

def make_vector(words):
    v = svector()
    for word in words:
        v[word] += 1
    return v
    
def test(devfile, model):
    tot, err = 0, 0
    for i, (label, words) in enumerate(read_from(devfile), 1): 
        err += label * (model.dot(make_vector(words))) <= 0
    return err/i  
            
def train(trainfile, devfile, epochs=5):
    t = time.time()
    best_err = 1.
    model = svector()
    for it in range(1, epochs+1):
        updates = 0
        for i, (label, words) in enumerate(read_from(trainfile), 1): 
            sent = make_vector(words)
            if label * (model.dot(sent)) <= 0:
                updates += 1
                model += label * sent
        dev_err = test(devfile, model)
        best_err = min(best_err, dev_err)
        print("epoch %d, update %.1f%%, dev %.1f%%" % (it, updates / i * 100, dev_err * 100))
    print("best dev err %.1f%%, |w|=%d, time: %.1f secs" % (best_err * 100, len(model), time.time() - t))


In [29]:
train('train.csv','dev.csv')

epoch 1, update 38.8%, dev 36.6%
epoch 2, update 25.1%, dev 34.6%
epoch 3, update 20.7%, dev 33.8%
epoch 4, update 16.7%, dev 31.7%
epoch 5, update 13.8%, dev 34.0%
best dev err 31.7%, |w|=16743, time: 1.0 secs


# Adding bias in make_vector function

In [30]:
from __future__ import division 

import sys
import time
import pandas as pd
from svector import svector

def read_from(textfile):
    data = pd.read_csv(textfile)
    for i in range(len(data)):
        id, words, label = data.iloc[i]
        yield (1 if label=="+" else -1, words.split())

def make_vector(words):
    v = svector()
    for word in words:
        v[word] += 1
    v["<bias>"] = 1
    return v
    
def test(devfile, model):
    tot, err = 0, 0
    for i, (label, words) in enumerate(read_from(devfile), 1): 
        err += label * (model.dot(make_vector(words))) <= 0
    return err/i  
            
def train(trainfile, devfile, epochs=5):
    t = time.time()
    best_err = 1.
    model = svector()
    for it in range(1, epochs+1):
        updates = 0
        for i, (label, words) in enumerate(read_from(trainfile), 1): 
            sent = make_vector(words)
            if label * (model.dot(sent)) <= 0:
                updates += 1
                model += label * sent
        dev_err = test(devfile, model)
        if dev_err < best_err:
            best_err = dev_err
            best_model = model.copy()
        print("epoch %d, update %.1f%%, dev %.1f%%" % (it, updates / i * 100, dev_err * 100))
    print("best dev err %.1f%%, |w|=%d, time: %.1f secs" % (best_err * 100, len(model), time.time() - t))
    return best_model


In [31]:
best_model = train('train.csv','dev.csv',10)

epoch 1, update 39.0%, dev 39.6%
epoch 2, update 25.5%, dev 34.1%
epoch 3, update 20.8%, dev 35.3%
epoch 4, update 17.2%, dev 35.5%
epoch 5, update 14.1%, dev 28.9%
epoch 6, update 12.2%, dev 32.0%
epoch 7, update 10.5%, dev 32.0%
epoch 8, update 9.7%, dev 31.5%
epoch 9, update 7.8%, dev 30.2%
epoch 10, update 6.9%, dev 29.8%
best dev err 28.9%, |w|=16744, time: 1.9 secs


In [32]:
def predict_test(testfile, model, output_file="test.predicted.csv"):
    test_data = pd.read_csv(testfile)
    predictions = []
    for i in range(len(test_data)):
        id, words = test_data.iloc[i, 0], test_data.iloc[i, 1]
        words_vector = make_vector(words.split())
        prediction = "+" if model.dot(words_vector) > 0 else "-"
        predictions.append(prediction)
    
    output_df = pd.DataFrame({
        'id': test_data['id'],
        'sentence': test_data['sentence'],
        'target': predictions
    })
    output_df.to_csv(output_file, index=False)
    print(f"Predictions saved to {output_file}")

In [33]:
predict_test('test.csv', best_model)

Predictions saved to test.predicted.csv


# Part 2: Average Perceptron 

In [34]:
import time

def train_with_optimized_averaging(trainfile, devfile, epochs=10):
    t = time.time()
    best_err = 1.0
    model = svector()  
    aux_vector = svector()  
    total_examples = 0  
    learning_rate = 1
    
    for it in range(1, epochs + 1):
        updates = 0
        for i, (label, words) in enumerate(read_from(trainfile), 1):
            sent = make_vector(words)
            if label * (model.dot(sent)) <= 0:
                updates += 1
                model += learning_rate * label * sent
                aux_vector += total_examples * learning_rate * label * sent
            
            total_examples += 1  
            
        averaged_model = svector()
        for key in model:
            averaged_model[key] = model[key] - (aux_vector[key] / total_examples)
        
        dev_err = test(devfile, averaged_model)
        best_err = min(best_err, dev_err)
        print("epoch %d, update %.1f%%, dev %.1f%%" % (it, updates / i * 100, dev_err * 100))

    print("best dev err %.1f%%, |w|=%d, time: %.1f secs" % (best_err * 100, len(averaged_model), time.time() - t))
    return averaged_model

In [35]:
averaged_model = train_with_optimized_averaging('train.csv','dev.csv',10)

epoch 1, update 39.0%, dev 31.4%
epoch 2, update 25.5%, dev 27.7%
epoch 3, update 20.8%, dev 27.2%
epoch 4, update 17.2%, dev 27.6%
epoch 5, update 14.1%, dev 27.2%
epoch 6, update 12.2%, dev 26.7%
epoch 7, update 10.5%, dev 26.3%
epoch 8, update 9.7%, dev 26.4%
epoch 9, update 7.8%, dev 26.3%
epoch 10, update 6.9%, dev 26.3%
best dev err 26.3%, |w|=16744, time: 2.1 secs


In [36]:
positive_features = sorted(averaged_model.items(), key=lambda x: x[1], reverse=True)[:20]
negative_features = sorted(averaged_model.items(), key=lambda x: x[1])[:20]

print("Top 20 Most Positive Features:")
for feature, weight in positive_features:
    print(f"{feature}: {weight}")

print("\nTop 20 Most Negative Features:")
for feature, weight in negative_features:
    print(f"{feature}: {weight}")

Top 20 Most Positive Features:
engrossing: 12.191025
triumph: 11.331724999999999
unexpected: 11.1279625
rare: 11.12445
provides: 10.9797625
french: 10.803925
skin: 10.62065
treat: 10.5876875
pulls: 10.4001875
culture: 10.2416625
cinema: 10.2386125
dots: 10.2072375
wonderful: 10.1634875
refreshingly: 10.05425
open: 9.895825
powerful: 9.7530875
delightful: 9.7265875
imax: 9.6073375
smarter: 9.382075
flaws: 9.379925

Top 20 Most Negative Features:
boring: -14.9132875
generic: -13.0597
dull: -12.8838125
badly: -11.867687499999999
routine: -11.713675
fails: -11.14415
ill: -11.0038625
too: -10.5788375
instead: -10.22245
tv: -10.2003125
attempts: -9.94605
unless: -9.9310375
incoherent: -9.856137499999999
neither: -9.846925
flat: -9.78285
seagal: -9.6624
problem: -9.631237500000001
scattered: -9.594899999999999
worst: -9.5857125
suffers: -9.571525


In [37]:
dev_data = pd.read_csv('dev.csv')

negative_as_positive = []
positive_as_negative = []

for i, row in dev_data.iterrows():
    label = row['target']  
    sentence = row['sentence']  
    words = sentence.split()
    vector = make_vector(words)
    score = averaged_model.dot(vector)
    
    if label == '-' and score > 0:
        negative_as_positive.append((sentence, score))
    
    elif label == '+' and score < 0:
        positive_as_negative.append((sentence, score))

negative_as_positive = sorted(negative_as_positive, key=lambda x: x[1], reverse=True)[:5]
positive_as_negative = sorted(positive_as_negative, key=lambda x: abs(x[1]), reverse=True)[:5]

print("5 Negative Examples Strongly Predicted as Positive:")
for sentence, score in negative_as_positive:
    print(f"Score: {score:.2f} | Sentence: {sentence}")

print("\n5 Positive Examples Strongly Predicted as Negative:")
for sentence, score in positive_as_negative:
    print(f"Score: {score:.2f} | Sentence: {sentence}")

5 Negative Examples Strongly Predicted as Positive:
Score: 35.11 | Sentence: ` in this poor remake of such a well loved classic , parker exposes the limitations of his skill and the basic flaws in his vision '
Score: 30.07 | Sentence: how much you are moved by the emotional tumult of fran ois and mich le 's relationship depends a lot on how interesting and likable you find them
Score: 29.72 | Sentence: bravo reveals the true intent of her film by carefully selecting interview subjects who will construct a portrait of castro so predominantly charitable it can only be seen as propaganda
Score: 28.71 | Sentence: mr wollter and ms seldhal give strong and convincing performances , but neither reaches into the deepest recesses of the character to unearth the quaking essence of passion , grief and fear
Score: 23.72 | Sentence: an atonal estrogen opera that demonizes feminism while gifting the most sympathetic male of the piece with a nice vomit bath at his wedding

5 Positive Examples Strongl

In [38]:
predict_test('test.csv', averaged_model,'test.predicted.p2.csv')

Predictions saved to test.predicted.p2.csv


# Part 3

In [39]:
from __future__ import division 

from collections import Counter
import pandas as pd
from svector import svector

def read_from(textfile):
    data = pd.read_csv(textfile)
    for i in range(len(data)):
        id, words, label = data.iloc[i]
        yield (1 if label=="+" else -1, words.split())

def make_vector(words, word_counts, min_freq):
    v = svector()
    for word in words:
        if word in word_counts and word_counts[word] >= min_freq:
            v[word] += 1
    v["<bias>"] = 1
    return v

def test(devfile, model, word_counts, min_freq):
    tot, err = 0, 0
    for i, (label, words) in enumerate(read_from(devfile), 1): 
        err += label * (model.dot(make_vector(words, word_counts, min_freq))) <= 0
    return err / i  

def get_word_frequencies(filename):
    data = pd.read_csv(filename)
    word_counts = {}
    for sentence in data['sentence']:
        words = sentence.split()
        for idx, word in enumerate(words):
            word_counts[word] = word_counts.get(word, 0) + 1
    return word_counts

word_counts = get_word_frequencies('train.csv')


In [40]:
import time

def train_with_optimized_averaging(trainfile, devfile, epochs=10, min_freq = 2):
    t = time.time()
    best_err = 1.0
    model = svector()  
    aux_vector = svector()  
    total_examples = 0  
    learning_rate = 1
    
    for it in range(1, epochs + 1):
        updates = 0
        for i, (label, words) in enumerate(read_from(trainfile), 1):
            sent = make_vector(words, word_counts, min_freq)
            if label * (model.dot(sent)) <= 0:
                updates += 1
                model += learning_rate * label * sent
                aux_vector += total_examples * learning_rate * label * sent  
            total_examples += 1  
        averaged_model = svector()
        for key in model:
            averaged_model[key] = model[key] - (aux_vector[key] / total_examples)
        
        dev_err = test(devfile, averaged_model, word_counts, min_freq)
        best_err = min(best_err, dev_err)
        print("epoch %d, update %.1f%%, dev %.1f%%" % (it, updates / i * 100, dev_err * 100))

    print("best dev err %.1f%%, |w|=%d, time: %.1f secs" % (best_err * 100, len(averaged_model), time.time() - t))
    return averaged_model

In [41]:
averaged_model_one_count = train_with_optimized_averaging('train.csv', 'dev.csv', epochs=10, min_freq=2)

epoch 1, update 39.0%, dev 31.6%
epoch 2, update 26.4%, dev 27.5%
epoch 3, update 22.8%, dev 26.8%
epoch 4, update 18.8%, dev 26.6%
epoch 5, update 17.2%, dev 25.9%
epoch 6, update 14.8%, dev 26.5%
epoch 7, update 13.2%, dev 27.0%
epoch 8, update 12.7%, dev 26.7%
epoch 9, update 11.4%, dev 26.6%
epoch 10, update 10.6%, dev 26.2%
best dev err 25.9%, |w|=8425, time: 2.2 secs


In [42]:
def predict_test(testfile, model, word_counts, min_freq, output_file="test.predicted.csv"):
    test_data = pd.read_csv(testfile)
    predictions = []
    for i in range(len(test_data)):
        id, words = test_data.iloc[i, 0], test_data.iloc[i, 1]
        words_vector = make_vector(words.split(), word_counts, min_freq)  
        prediction = "+" if model.dot(words_vector) > 0 else "-"
        predictions.append(prediction)
    
    output_df = pd.DataFrame({
        'id': test_data['id'],
        'sentence': test_data['sentence'],
        'target': predictions
    })
    output_df.to_csv(output_file, index=False)
    print(f"Predictions saved to {output_file}")

In [43]:
predict_test('test.csv', averaged_model_one_count, word_counts, min_freq=2, output_file="test.predicted.p3-1.csv")

Predictions saved to test.predicted.p3-1.csv


In [44]:
predict_test('dev.csv', averaged_model_one_count, word_counts, min_freq=2, output_file="dev-data-onehot-avg-perceptron.csv")

Predictions saved to dev-data-onehot-avg-perceptron.csv


In [45]:
averaged_model_two_count = train_with_optimized_averaging('train.csv', 'dev.csv', epochs=10, min_freq=3)

epoch 1, update 38.9%, dev 31.1%
epoch 2, update 28.2%, dev 29.2%
epoch 3, update 23.7%, dev 28.7%
epoch 4, update 21.7%, dev 28.0%
epoch 5, update 18.5%, dev 27.9%
epoch 6, update 17.7%, dev 26.6%
epoch 7, update 16.2%, dev 26.8%
epoch 8, update 15.2%, dev 26.6%
epoch 9, update 14.1%, dev 26.6%
epoch 10, update 12.6%, dev 26.6%
best dev err 26.6%, |w|=5934, time: 2.2 secs


In [46]:
predict_test('test.csv', averaged_model_two_count, word_counts, min_freq=3, output_file="test.predicted.p3-2.csv")

Predictions saved to test.predicted.p3-2.csv


# Part 4 - Using logistic regression

In [47]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction import DictVectorizer
import numpy as np
import pandas as pd

def get_word_frequencies(filename):
    data = pd.read_csv(filename)
    word_counts = {}
    for sentence in data['sentence']:
        words = sentence.split()
        for idx, word in enumerate(words):
            word_counts[word] = word_counts.get(word, 0) + 1

    return word_counts

def get_pruned_vocabulary(trainfile, min_freq=2):
    word_counts = get_word_frequencies(trainfile)
    return {word for word, count in word_counts.items() if count >= min_freq}, word_counts

def make_vector(words, word_counts, min_freq=2):
    v = {}
    # Unigrams
    for word in words:
        if word_counts.get(word, 0) >= min_freq:
            v[word] = v.get(word, 0) + 1
    v["<bias>"] = 1
    return v

def vectorize_data(filename, word_counts, min_freq=2):
    data = pd.read_csv(filename)
    feature_vectors = []
    for sentence in data['sentence']:
        words = sentence.split()
        v = make_vector(words, word_counts, min_freq)
        feature_vectors.append(v)
    y = data['target'].apply(lambda x: 1 if x == "+" else 0).values
    return feature_vectors, y

def train_logistic_regression(trainfile, devfile, min_freq=2):
    t = time.time()
    vocabulary, word_counts = get_pruned_vocabulary(trainfile, min_freq)
    X_train_feats, y_train = vectorize_data(trainfile, word_counts, min_freq)
    X_dev_feats, y_dev = vectorize_data(devfile, word_counts, min_freq)

    vectorizer = DictVectorizer(sparse=True)
    X_train = vectorizer.fit_transform(X_train_feats)
    X_dev = vectorizer.transform(X_dev_feats)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    dev_predictions = model.predict(X_dev)
    dev_error = 1 - accuracy_score(y_dev, dev_predictions)
    
    print("best dev err %.1f%%, time: %.1f secs" % (dev_error * 100, time.time() - t))
    return model, vectorizer,word_counts

trainfile = 'train.csv'
devfile = 'dev.csv'
model, vectorizer,word_counts = train_logistic_regression(trainfile, devfile, min_freq=2)

best dev err 25.7%, time: 0.2 secs


In [48]:
def predict_test_logistic_regression(testfile, model, vectorizer, word_counts, min_freq=2, output_file="test.predicted.csv"):
 
    data = pd.read_csv(testfile)
    feature_vectors = []
    for sentence in data['sentence']:
        words = sentence.split()
        v = make_vector(words, word_counts, min_freq)
        feature_vectors.append(v)
    
    X_test = vectorizer.transform(feature_vectors)
    
    predictions = model.predict(X_test)
    predictions = ["+" if pred == 1 else "-" for pred in predictions]
    
    output_df = pd.DataFrame({
        'id': data['id'],
        'sentence': data['sentence'],
        'target': predictions
    })
    output_df.to_csv(output_file, index=False)
    print(f"Predictions saved to {output_file}")

In [49]:
trainfile = 'train.csv'
devfile = 'dev.csv'
testfile = 'test.csv'
output_file = 'test.predicted.p4.csv'

model, vectorizer, word_counts = train_logistic_regression(trainfile, devfile, min_freq=2)
predict_test_logistic_regression(testfile, model, vectorizer, word_counts, min_freq=2, output_file=output_file)

best dev err 25.7%, time: 0.2 secs
Predictions saved to test.predicted.p4.csv


# Part 5 - Optimizing logistic regression

In [50]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction import DictVectorizer
import numpy as np
import pandas as pd

def get_word_frequencies(filename):
    data = pd.read_csv(filename)
    word_counts = {}
    for sentence in data['sentence']:
        words = sentence.split()
        for idx, word in enumerate(words):
            word_counts[word] = word_counts.get(word, 0) + 1
            if idx < len(words) - 1:
                bigram = f"{word} {words[idx+1]}"
                word_counts[bigram] = word_counts.get(bigram, 0) + 1
    return word_counts

def get_pruned_vocabulary(trainfile, min_freq=2):
    word_counts = get_word_frequencies(trainfile)
    return {word for word, count in word_counts.items() if count >= min_freq}, word_counts

def make_vector(words, word_counts, min_freq=2):
    v = {}
    # Unigrams
    for word in words:
        if word_counts.get(word, 0) >= min_freq:
            v[word] = v.get(word, 0) + 1
    # Bigrams
    for i in range(len(words) - 1):
        bigram = f"{words[i]} {words[i+1]}"
        if word_counts.get(bigram, 0) >= min_freq:
            v[bigram] = v.get(bigram, 0) + 1
    v["<bias>"] = 1
    return v

def vectorize_data(filename, word_counts, min_freq=2):
    data = pd.read_csv(filename)
    feature_vectors = []
    for sentence in data['sentence']:
        words = sentence.split()
        v = make_vector(words, word_counts, min_freq)
        feature_vectors.append(v)
    y = data['target'].apply(lambda x: 1 if x == "+" else 0).values
    return feature_vectors, y

def train_logistic_regression_optimized(trainfile, devfile, min_freq=2):
    t = time.time()
    vocabulary, word_counts = get_pruned_vocabulary(trainfile, min_freq)
    X_train_feats, y_train = vectorize_data(trainfile, word_counts, min_freq)
    X_dev_feats, y_dev = vectorize_data(devfile, word_counts, min_freq)

    vectorizer = DictVectorizer(sparse=True)
    X_train = vectorizer.fit_transform(X_train_feats)
    X_dev = vectorizer.transform(X_dev_feats)

    model = LogisticRegression(max_iter=1000, C=2)
    model.fit(X_train, y_train)
    dev_predictions = model.predict(X_dev)
    dev_error = 1 - accuracy_score(y_dev, dev_predictions)
    
    print("best dev err %.1f%%, time: %.1f secs" % (dev_error * 100, time.time() - t))
    return model, vectorizer,word_counts

trainfile = 'train.csv'
devfile = 'dev.csv'
model, vectorizer,word_counts = train_logistic_regression_optimized(trainfile, devfile, min_freq=2)

best dev err 24.6%, time: 1.8 secs


In [51]:
trainfile = 'train.csv'
devfile = 'dev.csv'
testfile = 'test.csv'
output_file = 'test.predicted.p5.csv'

model, vectorizer, word_counts = train_logistic_regression_optimized(trainfile, devfile, min_freq=2)
predict_test_logistic_regression(testfile, model, vectorizer, word_counts, min_freq=2, output_file=output_file)

best dev err 24.6%, time: 1.9 secs
Predictions saved to test.predicted.p5.csv
